# Generate S3 traffic for InstantEvidence

Use this notebook to create S3 activity in your bucket—uploads, reads, copies, listings, and deletes—so you can confirm **InstantEvidence** is receiving and evaluating events the way you expect.

Operations run through the [AWS API MCP Server](https://github.com/awslabs/mcp/tree/main/src/aws-api-mcp-server), which executes standard AWS S3 API calls using the credentials you provide.

Object keys use mixed prefixes (for example `public/`, `data/`, and `secret/`) so InstantEvidence can exercise allow/deny style rules during demos.

## Before you begin

### AWS credentials

In Google Colab, open **Secrets** (key icon in the left sidebar) and add:

| Secret | Required |
|--------|----------|
| `AWS_ACCESS_KEY_ID` | Yes |
| `AWS_SECRET_ACCESS_KEY` | Yes |
| `AWS_REGION` | No — defaults to `us-east-1` |
| `AWS_SESSION_TOKEN` | No — only if your credentials include a session token (SSO or assumed role) |

Use credentials scoped to your **test bucket only**. Minimum IAM actions:

- `sts:GetCallerIdentity`
- `s3:HeadBucket`, `s3:ListBucket` on the bucket
- `s3:PutObject`, `s3:GetObject`, `s3:HeadObject`, `s3:DeleteObject`, `s3:CopyObject` on `arn:aws:s3:::YOUR_BUCKET/*`

Avoid production admin keys.

### Your bucket and InstantEvidence

- Set **BUCKET** in the run cell to the same S3 bucket InstantEvidence monitors.
- Ensure object events from that bucket reach InstantEvidence (for example via SNS or EventBridge to your S3 event webhook).

## How to run

1. Open this notebook in [Google Colab](https://colab.research.google.com/).
2. Add the secrets above.
3. Set **BUCKET** to your bucket name and adjust **CYCLES** / intervals if needed.
4. Select **Runtime → Run all**.

The traffic loop usually takes a few minutes (default 20 cycles with pauses between operations). If it stops early after repeated errors, check the bucket name, secrets, and IAM permissions.

All steps are in this notebook—nothing else to install or upload.


In [ ]:
%pip install -q awslabs.aws-api-mcp-server mcp


In [ ]:
"""S3 traffic generator — credentials, MCP client, and traffic loop (single cell)."""

from __future__ import annotations


import os
from dataclasses import dataclass

AWS_ACCESS_KEY_ID = "AWS_ACCESS_KEY_ID"
AWS_SECRET_ACCESS_KEY = "AWS_SECRET_ACCESS_KEY"
AWS_REGION = "AWS_REGION"
AWS_SESSION_TOKEN = "AWS_SESSION_TOKEN"
DEFAULT_REGION = "us-east-1"


class CredentialError(Exception):
    """Raised when AWS credentials cannot be resolved."""


@dataclass(frozen=True)
class AwsCredentials:
    access_key_id: str
    secret_access_key: str
    region: str
    session_token: str | None = None

    def to_env(self) -> dict[str, str]:
        env = {
            AWS_ACCESS_KEY_ID: self.access_key_id,
            AWS_SECRET_ACCESS_KEY: self.secret_access_key,
            AWS_REGION: self.region,
            "AWS_DEFAULT_REGION": self.region,
        }
        if self.session_token:
            env[AWS_SESSION_TOKEN] = self.session_token
        return env


def _colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        value = userdata.get(name)
    except Exception:
        return None
    return value.strip() if value else None


def _lookup(name: str) -> str | None:
    value = _colab_secret(name) or os.getenv(name)
    return value.strip() if value else None


def resolve_aws_credentials() -> AwsCredentials:
    """Load AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY from Colab secrets or env."""
    access_key_id = _lookup(AWS_ACCESS_KEY_ID)
    secret_access_key = _lookup(AWS_SECRET_ACCESS_KEY)
    region = _lookup(AWS_REGION) or DEFAULT_REGION
    session_token = _lookup(AWS_SESSION_TOKEN)

    if not access_key_id or not secret_access_key:
        raise CredentialError(
            "Set Colab secrets (or environment variables):\n"
            f"  • {AWS_ACCESS_KEY_ID}\n"
            f"  • {AWS_SECRET_ACCESS_KEY}\n"
            f"Optional: {AWS_REGION} (default {DEFAULT_REGION}), {AWS_SESSION_TOKEN}"
        )

    return AwsCredentials(
        access_key_id=access_key_id,
        secret_access_key=secret_access_key,
        region=region,
        session_token=session_token,
    )


def mask_access_key(access_key_id: str) -> str:
    if len(access_key_id) <= 8:
        return "***"
    return f"{access_key_id[:4]}...{access_key_id[-4:]}"




import asyncio
import json
import os
import shlex
import sys
import uuid
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import quote

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


MCP_CONNECT_TIMEOUT_SEC = 20.0
MCP_CALL_TIMEOUT_SEC = 15.0


class McpS3Error(Exception):
    """Raised when an MCP S3 operation fails."""


@dataclass
class S3OperationResult:
    command: str
    response: dict[str, Any] | None = None
    data: Any = None
    error: str | None = None

    @property
    def ok(self) -> bool:
        return self.error is None


def format_copy_source(bucket: str, source_key: str) -> str:
    """URL-encode the key portion for ``aws s3api copy-object --copy-source``."""
    return f"{bucket}/{quote(source_key, safe='/')}"


def _region_flag(region: str) -> str:
    return f"--region {shlex.quote(region)}"


def build_head_bucket_command(bucket: str, region: str) -> str:
    return f"aws s3api head-bucket --bucket {shlex.quote(bucket)} {_region_flag(region)}"


def build_list_buckets_command(region: str) -> str:
    return f"aws s3api list-buckets {_region_flag(region)}"


def build_list_objects_command(
    bucket: str,
    region: str,
    *,
    prefix: str = "",
    max_keys: int = 100,
) -> str:
    cmd = (
        f"aws s3api list-objects-v2 --bucket {shlex.quote(bucket)} "
        f"--max-keys {int(max_keys)} {_region_flag(region)}"
    )
    if prefix:
        cmd += f" --prefix {shlex.quote(prefix)}"
    return cmd


def build_put_object_command(
    bucket: str,
    key: str,
    body_path: Path | str,
    region: str,
    *,
    content_type: str = "text/plain",
) -> str:
    return (
        f"aws s3api put-object --bucket {shlex.quote(bucket)} "
        f"--key {shlex.quote(key)} --body {shlex.quote(str(body_path))} "
        f"--content-type {shlex.quote(content_type)} {_region_flag(region)}"
    )


def build_get_object_command(bucket: str, key: str, out_path: Path | str, region: str) -> str:
    return (
        f"aws s3api get-object --bucket {shlex.quote(bucket)} "
        f"--key {shlex.quote(key)} {shlex.quote(str(out_path))} {_region_flag(region)}"
    )


def build_head_object_command(bucket: str, key: str, region: str) -> str:
    return (
        f"aws s3api head-object --bucket {shlex.quote(bucket)} "
        f"--key {shlex.quote(key)} {_region_flag(region)}"
    )


def build_copy_object_command(bucket: str, source_key: str, dest_key: str, region: str) -> str:
    copy_source = format_copy_source(bucket, source_key)
    return (
        f"aws s3api copy-object --bucket {shlex.quote(bucket)} "
        f"--copy-source {shlex.quote(copy_source)} "
        f"--key {shlex.quote(dest_key)} {_region_flag(region)}"
    )


def build_delete_object_command(bucket: str, key: str, region: str) -> str:
    return (
        f"aws s3api delete-object --bucket {shlex.quote(bucket)} "
        f"--key {shlex.quote(key)} {_region_flag(region)}"
    )


def extract_call_aws_payload(result: Any) -> list[dict[str, Any]]:
    if getattr(result, "structuredContent", None):
        payload = result.structuredContent
        if isinstance(payload, dict) and "result" in payload:
            return payload["result"]
    for block in getattr(result, "content", []) or []:
        text = getattr(block, "text", None)
        if not text:
            continue
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            return [parsed]
    return []


def unwrap_aws_data(response: Any) -> Any | None:
    """Extract the AWS API JSON payload from an MCP ``call_aws`` wrapper."""
    if response is None:
        return None
    if isinstance(response, str):
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            return response
    if not isinstance(response, dict):
        return response

    inner = response.get("response")
    if isinstance(inner, dict):
        raw_json = inner.get("json") or inner.get("as_json")
        if raw_json:
            try:
                return json.loads(raw_json)
            except (json.JSONDecodeError, TypeError):
                return raw_json
    return response


def parse_call_aws_row(cli_command: str, row: dict[str, Any]) -> S3OperationResult:
    error = row.get("error")
    if error:
        return S3OperationResult(command=cli_command, error=str(error))

    raw_response = row.get("response")
    if isinstance(raw_response, dict):
        inner = raw_response.get("response")
        if isinstance(inner, dict) and inner.get("error"):
            return S3OperationResult(
                command=cli_command,
                error=str(inner["error"]),
                response=raw_response,
            )

    data = unwrap_aws_data(raw_response)
    response_dict = raw_response if isinstance(raw_response, dict) else {"raw": raw_response}
    return S3OperationResult(command=cli_command, response=response_dict, data=data)


class AwsMcpS3Client:
    """Thin wrapper around AWS API MCP server for S3 object CRUD."""

    def __init__(
        self,
        credentials: AwsCredentials,
        *,
        read_only: bool = False,
        region: str | None = None,
        workdir: Path | None = None,
        call_timeout_sec: float = MCP_CALL_TIMEOUT_SEC,
    ) -> None:
        self.credentials = credentials
        self.read_only = read_only
        self.region = region or credentials.region
        self._workdir = workdir or self._default_workdir()
        self._call_timeout_sec = call_timeout_sec
        self._session: ClientSession | None = None
        self._transport = None

    @staticmethod
    def _default_workdir() -> Path:
        path = Path(os.environ.get("TMPDIR", "/tmp")) / "aws-api-mcp" / "workdir"
        path.mkdir(parents=True, exist_ok=True)
        return path

    def _server_env(self) -> dict[str, str]:
        env = self.credentials.to_env()
        env["READ_OPERATIONS_ONLY"] = "true" if self.read_only else "false"
        env["AWS_API_MCP_TELEMETRY"] = "false"
        env["AWS_API_MCP_WORKING_DIR"] = str(self._workdir)
        return env

    def _write_temp_file(self, body: str, *, suffix: str = ".txt") -> Path:
        name = f"mcp-s3-{uuid.uuid4().hex}{suffix}"
        path = self._workdir / name
        fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
        with os.fdopen(fd, "w", encoding="utf-8") as handle:
            handle.write(body)
        return path

    async def __aenter__(self) -> AwsMcpS3Client:
        params = StdioServerParameters(
            command=sys.executable,
            args=["-m", "awslabs.aws_api_mcp_server.server"],
            env=self._server_env(),
        )
        self._transport = stdio_client(params)
        try:
            read, write = await asyncio.wait_for(
                self._transport.__aenter__(),
                timeout=MCP_CONNECT_TIMEOUT_SEC,
            )
        except Exception:
            await self._cleanup_transport(None, None, None)
            raise

        self._session = ClientSession(read, write)
        try:
            await asyncio.wait_for(
                self._session.__aenter__(),
                timeout=MCP_CONNECT_TIMEOUT_SEC,
            )
            await asyncio.wait_for(
                self._session.initialize(),
                timeout=MCP_CONNECT_TIMEOUT_SEC,
            )
        except Exception as exc:
            await self._cleanup_session(None, exc, None)
            await self._cleanup_transport(None, exc, None)
            raise
        return self

    async def _cleanup_session(self, exc_type, exc, tb) -> None:
        if self._session is not None:
            await self._session.__aexit__(exc_type, exc, tb)
            self._session = None

    async def _cleanup_transport(self, exc_type, exc, tb) -> None:
        if self._transport is not None:
            await self._transport.__aexit__(exc_type, exc, tb)
            self._transport = None

    async def __aexit__(self, exc_type, exc, tb) -> None:
        await self._cleanup_session(exc_type, exc, tb)
        await self._cleanup_transport(exc_type, exc, tb)

    async def aclose(self) -> None:
        """Explicit shutdown for notebook interrupt handlers."""
        await self.__aexit__(None, None, None)

    async def call_aws(self, cli_command: str, *, max_results: int | None = None) -> S3OperationResult:
        if self._session is None:
            raise McpS3Error("Client is not connected. Use async with AwsMcpS3Client(...).")

        arguments: dict[str, Any] = {"cli_command": cli_command}
        if max_results is not None:
            arguments["max_results"] = max_results

        try:
            result = await asyncio.wait_for(
                self._session.call_tool("call_aws", arguments=arguments),
                timeout=self._call_timeout_sec,
            )
        except TimeoutError:
            return S3OperationResult(
                command=cli_command,
                error=f"MCP call timed out after {self._call_timeout_sec:.0f}s",
            )

        rows = extract_call_aws_payload(result)
        if not rows:
            return S3OperationResult(command=cli_command, error="Empty MCP response")
        return parse_call_aws_row(cli_command, rows[0])

    async def head_bucket(self, bucket: str) -> S3OperationResult:
        return await self.call_aws(build_head_bucket_command(bucket, self.region))

    async def list_buckets(self) -> S3OperationResult:
        return await self.call_aws(build_list_buckets_command(self.region))

    async def list_objects(
        self,
        bucket: str,
        *,
        prefix: str = "",
        max_keys: int = 100,
    ) -> S3OperationResult:
        return await self.call_aws(
            build_list_objects_command(bucket, self.region, prefix=prefix, max_keys=max_keys)
        )

    async def put_object(
        self,
        bucket: str,
        key: str,
        body: str,
        *,
        content_type: str = "text/plain",
    ) -> S3OperationResult:
        payload_path = self._write_temp_file(body)
        try:
            cmd = build_put_object_command(
                bucket, key, payload_path, self.region, content_type=content_type
            )
            return await self.call_aws(cmd)
        finally:
            payload_path.unlink(missing_ok=True)

    async def get_object(self, bucket: str, key: str) -> S3OperationResult:
        out_path = self._workdir / f"mcp-s3-get-{uuid.uuid4().hex}.bin"
        try:
            result = await self.call_aws(
                build_get_object_command(bucket, key, out_path, self.region)
            )
            if result.ok and out_path.exists():
                preview = out_path.read_bytes()[:500].decode("utf-8", errors="replace")
                result.data = {**(result.data or {}), "BodyPreview": preview}
            return result
        finally:
            out_path.unlink(missing_ok=True)

    async def head_object(self, bucket: str, key: str) -> S3OperationResult:
        return await self.call_aws(build_head_object_command(bucket, key, self.region))

    async def copy_object(
        self,
        bucket: str,
        source_key: str,
        dest_key: str,
    ) -> S3OperationResult:
        return await self.call_aws(
            build_copy_object_command(bucket, source_key, dest_key, self.region)
        )

    async def delete_object(self, bucket: str, key: str) -> S3OperationResult:
        return await self.call_aws(build_delete_object_command(bucket, key, self.region))



import asyncio
import random
import uuid
from collections import deque
from contextlib import asynccontextmanager
from dataclasses import dataclass, field
from datetime import UTC, datetime
from typing import AsyncIterator


# Demo keys span allow- and deny-style prefixes for InstantEvidence policy checks.
KEY_PREFIXES_ALLOWED = ("public/", "data/", "uploads/", "archive/")
KEY_PREFIXES_DENIED = ("secret/", "tmp/", "temp/", "staging/")
KEY_PREFIXES = KEY_PREFIXES_ALLOWED + KEY_PREFIXES_DENIED
KEY_SUFFIXES = (".csv", ".json", ".txt", ".log", ".pdf", "")

OPERATIONS = ("PutObject", "GetObject", "HeadObject", "ListObjectsV2", "CopyObject", "DeleteObject")
OPERATION_WEIGHTS = (35, 15, 10, 10, 15, 15)
MAX_RECENT_KEYS = 50
MAX_CONSECUTIVE_ERRORS = 5
ERROR_BACKOFF_CAP_SEC = 8.0


@dataclass
class TrafficStats:
    started_at: datetime = field(default_factory=lambda: datetime.now(UTC))
    operations: dict[str, int] = field(default_factory=lambda: {op: 0 for op in OPERATIONS})
    errors: int = 0
    cycles: int = 0

    def record(self, operation: str, *, ok: bool) -> None:
        self.operations[operation] = self.operations.get(operation, 0) + 1
        if not ok:
            self.errors += 1


def random_key() -> str:
    prefix = random.choice(KEY_PREFIXES)
    suffix = random.choice(KEY_SUFFIXES)
    token = uuid.uuid4().hex[:8]
    return f"{prefix}traffic-{token}{suffix}"


def pick_operation(recent_keys: deque[str], *, read_only: bool = False) -> str:
    if read_only:
        if not recent_keys:
            return "ListObjectsV2"
        return random.choices(READ_OPERATIONS, weights=READ_OPERATION_WEIGHTS, k=1)[0]
    if not recent_keys:
        return "PutObject"
    return random.choices(OPERATIONS, weights=OPERATION_WEIGHTS, k=1)[0]


def error_backoff_sec(consecutive_errors: int) -> float:
    if consecutive_errors <= 0:
        return 0.0
    return min(2 ** (consecutive_errors - 1), ERROR_BACKOFF_CAP_SEC)


async def verify_access(client: AwsMcpS3Client, bucket: str) -> None:
    identity = await client.call_aws("aws sts get-caller-identity")
    if not identity.ok:
        raise McpS3Error(f"Credential check failed: {identity.error}")

    head = await client.head_bucket(bucket)
    if not head.ok:
        raise McpS3Error(f"Bucket access check failed for {bucket!r}: {head.error}")


async def execute_operation(
    client: AwsMcpS3Client,
    bucket: str,
    operation: str,
    recent_keys: deque[str],
    *,
    read_only: bool = False,
) -> tuple[str, S3OperationResult]:
    if operation == "PutObject":
        key = random_key()
        body = f"colab-traffic {datetime.now(UTC).isoformat()}"
        result = await client.put_object(bucket, key, body)
        if result.ok:
            recent_keys.append(key)
            while len(recent_keys) > MAX_RECENT_KEYS:
                recent_keys.popleft()
        return operation, result

    if operation == "ListObjectsV2":
        prefix = random.choice(KEY_PREFIXES)
        return operation, await client.list_objects(bucket, prefix=prefix)

    if not recent_keys:
        if read_only:
            prefix = random.choice(KEY_PREFIXES)
            return "ListObjectsV2", await client.list_objects(bucket, prefix=prefix)
        key = random_key()
        body = f"seed {datetime.now(UTC).isoformat()}"
        seed = await client.put_object(bucket, key, body)
        if seed.ok:
            recent_keys.append(key)
        return "PutObject", seed

    key = random.choice(list(recent_keys))

    if operation == "GetObject":
        return operation, await client.get_object(bucket, key)
    if operation == "HeadObject":
        return operation, await client.head_object(bucket, key)
    if operation == "CopyObject":
        dest_key = random_key()
        result = await client.copy_object(bucket, key, dest_key)
        if result.ok:
            recent_keys.append(dest_key)
        return operation, result
    if operation == "DeleteObject":
        result = await client.delete_object(bucket, key)
        if result.ok:
            try:
                recent_keys.remove(key)
            except ValueError:
                pass
        return operation, result

    raise McpS3Error(f"Unsupported operation: {operation}")


async def _run_cycles(
    client: AwsMcpS3Client,
    bucket: str,
    stats: TrafficStats,
    *,
    cycles: int,
    min_interval_sec: float,
    max_interval_sec: float,
    max_consecutive_errors: int,
    recent_keys: deque[str],
    read_only: bool = False,
) -> None:
    consecutive_errors = 0

    for cycle in range(1, cycles + 1):
        operation = pick_operation(recent_keys, read_only=read_only)
        op_name, result = await execute_operation(
            client, bucket, operation, recent_keys, read_only=read_only
        )
        stats.record(op_name, ok=result.ok)
        stats.cycles = cycle

        if result.ok:
            consecutive_errors = 0
            status = "OK"
        else:
            consecutive_errors += 1
            status = f"ERR: {result.error}"
            backoff = error_backoff_sec(consecutive_errors)
            if backoff:
                print(f"  backoff {backoff:.0f}s after error")
                await asyncio.sleep(backoff)
            if consecutive_errors >= max_consecutive_errors:
                raise McpS3Error(
                    f"Stopping after {consecutive_errors} consecutive errors. Last: {result.error}"
                )

        print(f"[{cycle}/{cycles}] {op_name} -> {status}")

        if cycle < cycles:
            delay = random.uniform(min_interval_sec, max_interval_sec)
            await asyncio.sleep(delay)


@asynccontextmanager
async def mcp_client(
    credentials: AwsCredentials,
    *,
    read_only: bool = False,
    region: str | None = None,
) -> AsyncIterator[AwsMcpS3Client]:
    client = AwsMcpS3Client(credentials, read_only=read_only, region=region)
    try:
        await client.__aenter__()
        yield client
    finally:
        await client.aclose()


async def run_traffic_loop(
    *,
    bucket: str,
    credentials: AwsCredentials | None = None,
    client: AwsMcpS3Client | None = None,
    cycles: int = 20,
    min_interval_sec: float = 3.0,
    max_interval_sec: float = 12.0,
    read_only: bool = False,
    region: str | None = None,
    max_consecutive_errors: int = MAX_CONSECUTIVE_ERRORS,
    verify: bool = True,
) -> TrafficStats:
    creds = credentials or resolve_aws_credentials()
    stats = TrafficStats()
    recent_keys: deque[str] = deque(maxlen=MAX_RECENT_KEYS)

    print(
        f"Starting MCP S3 traffic generator | bucket={bucket} region={region or creds.region} "
        f"key={mask_access_key(creds.access_key_id)} cycles={cycles} read_only={read_only}"
    )

    if client is not None:
        if verify:
            await verify_access(client, bucket)
            print("Caller identity and bucket access OK")
        await _run_cycles(
            client,
            bucket,
            stats,
            cycles=cycles,
            min_interval_sec=min_interval_sec,
            max_interval_sec=max_interval_sec,
            max_consecutive_errors=max_consecutive_errors,
            recent_keys=recent_keys,
            read_only=read_only,
        )
    else:
        async with mcp_client(creds, read_only=read_only, region=region) as managed:
            if verify:
                await verify_access(managed, bucket)
                print("Caller identity and bucket access OK")
            await _run_cycles(
                managed,
                bucket,
                stats,
                cycles=cycles,
                min_interval_sec=min_interval_sec,
                max_interval_sec=max_interval_sec,
                max_consecutive_errors=max_consecutive_errors,
                recent_keys=recent_keys,
                read_only=read_only,
            )

    print(
        f"Done. ops={stats.operations} errors={stats.errors} "
        f"elapsed={(datetime.now(UTC) - stats.started_at).total_seconds():.1f}s"
    )
    return stats



## Run traffic

Adjust the settings below, then run the cell. You will see one line per operation (for example `PutObject -> OK`). When the loop finishes, the cell output includes a summary of operations and any errors.

| Setting | What it does |
|---------|----------------|
| **BUCKET** | S3 bucket InstantEvidence monitors |
| **CYCLES** | How many operations to run |
| **MIN_INTERVAL_SEC** / **MAX_INTERVAL_SEC** | Random pause between operations (seconds) |
| **READ_ONLY** | When enabled, only list/read/head operations run (no uploads, copies, or deletes) |


In [ ]:
BUCKET = "your-instantevidence-bucket"  # @param {type:"string"}
CYCLES = 20  # @param {type:"integer"}
MIN_INTERVAL_SEC = 3.0  # @param {type:"number"}
MAX_INTERVAL_SEC = 12.0  # @param {type:"number"}
READ_ONLY = False  # @param {type:"boolean"}

try:
    creds = resolve_aws_credentials()
    print(f"Using {mask_access_key(creds.access_key_id)} in {creds.region}")
except CredentialError as exc:
    raise SystemExit(exc) from exc

mcp = AwsMcpS3Client(creds, read_only=READ_ONLY)
stats = None
try:
    await mcp.__aenter__()
    await verify_access(mcp, BUCKET)
    print("Smoke test OK")
    stats = await run_traffic_loop(
        bucket=BUCKET,
        credentials=creds,
        client=mcp,
        cycles=CYCLES,
        min_interval_sec=MIN_INTERVAL_SEC,
        max_interval_sec=MAX_INTERVAL_SEC,
        read_only=READ_ONLY,
        verify=False,
    )
finally:
    await mcp.aclose()

stats



## View results in InstantEvidence

If your bucket is connected to InstantEvidence, you should see new S3 events in the console shortly after each operation completes. If nothing appears, confirm event delivery (SNS or EventBridge) is configured for the bucket you set in **BUCKET**.
